# AGMS-Net Phase 1: Adaptive Scale Gating

**Change from Phase 0:** The static `nn.Linear` scale mapping in `MSPBlock` has been replaced with an `AdaptiveGate` module that dynamically weights the 5 temporal scales using `Softmax(MLP(GAP(scales)))`.

**Runtime**: GPU required (Runtime > Change runtime type > T4 GPU)

**Dataset**: Same `alibaba.csv` (96 machines, 11206 timesteps)

In [ ]:
# 1. Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Clone repo (fresh pull with Phase 1 AdaptiveGate code)
import os
!rm -rf /content/Workload_Prediction_MTP
!git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git /content/Workload_Prediction_MTP
%cd /content/Workload_Prediction_MTP/MSCNet_baseline
print('run.py exists:', os.path.exists('run.py'))

# Verify AdaptiveGate is present
with open('models/MSCNet.py') as f:
    content = f.read()
    assert 'AdaptiveGate' in content, 'ERROR: AdaptiveGate not found! Did you push the Phase 1 code?'
    print('AdaptiveGate module found in MSCNet.py')

In [ ]:
# 3. Install dependencies
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

In [ ]:
# 4. Upload dataset
import os, pandas as pd
os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/alibaba.csv'):
    from google.colab import files
    print('Upload alibaba.csv (96 machines, ~11MB):')
    uploaded = files.upload()
    for f in uploaded: os.rename(f, 'dataset/alibaba.csv')
df = pd.read_csv('dataset/alibaba.csv')
n_features = len(df.columns) - 1  # exclude date
print(f'Dataset: {df.shape}, Features: {n_features}')
print(f'Time: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')

In [ ]:
# 5. Setup
import os
os.makedirs('checkpoints', exist_ok=True)

import pandas as pd
df = pd.read_csv('dataset/alibaba.csv')
N_FEATURES = len(df.columns) - 1  # exclude 'date'
print(f'enc_in = dec_in = c_out = {N_FEATURES}')
print(f'Ready to train Phase 1 (AdaptiveGate)!')

In [ ]:
# 6. Train H=24
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_24 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 24 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 7. Train H=48
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_48 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 48 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 8. Train H=72
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_72 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 72 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 9. Train H=96
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_96 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 96 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 10. Results — Phase 1 vs Phase 0 Baseline
import numpy as np, glob, os

# Phase 0 baseline results (our reproduction)
phase0 = {
    24: {'MSE': 0.4630, 'MAE': 0.4927},
    48: {'MSE': 0.5638, 'MAE': 0.5438},
    72: {'MSE': 0.6447, 'MAE': 0.5794},
    96: {'MSE': 0.7213, 'MAE': 0.6144},
}

# Paper's original results
paper = {
    24: {'MSE': 0.5687, 'MAE': 0.4565},
    48: {'MSE': 0.6274, 'MAE': 0.4986},
    72: {'MSE': 0.6622, 'MAE': 0.5215},
    96: {'MSE': 0.6893, 'MAE': 0.5379},
}

print('=' * 105)
print('AGMS-Net Phase 1 (AdaptiveGate) vs Phase 0 Baseline vs Paper (Alibaba, L=96)')
print('=' * 105)
print(f'{"H":>4} | {"Phase1 MSE":>10} | {"Phase0 MSE":>10} | {"vs P0":>7} | {"Paper MSE":>10} | {"Phase1 MAE":>10} | {"Phase0 MAE":>10} | {"vs P0":>7}')
print('-' * 105)

for d in sorted(glob.glob('results/*')):
    mp = os.path.join(d, 'metrics.npy')
    if os.path.exists(mp):
        m = np.load(mp)
        mae, mse = m[0], m[1]
        for h in [24, 48, 72, 96]:
            if f'pl{h}_' in d:
                p0 = phase0[h]
                pp = paper[h]
                mse_delta = (mse - p0['MSE']) / p0['MSE'] * 100
                mae_delta = (mae - p0['MAE']) / p0['MAE'] * 100
                print(f'{h:>4} | {mse:>10.4f} | {p0["MSE"]:>10.4f} | {mse_delta:>+6.1f}% | {pp["MSE"]:>10.4f} | {mae:>10.4f} | {p0["MAE"]:>10.4f} | {mae_delta:>+6.1f}%')
                break
print('=' * 105)

In [ ]:
# 11. Print raw log
import os
if os.path.exists('result_long_term_forecast.txt'):
    print('--- Raw Results Log ---')
    with open('result_long_term_forecast.txt') as f: print(f.read())

In [ ]:
# 12. Gate Weight Analysis — Are the gates actually learning?
import torch, sys, numpy as np
sys.path.insert(0, '.')
from models.MSCNet import Model
import argparse, glob

# Find a checkpoint
ckpts = sorted(glob.glob('checkpoints/*/checkpoint.pth'))
if not ckpts:
    print('No checkpoints found!')
else:
    print(f'Found {len(ckpts)} checkpoint(s)')
    for ckpt_path in ckpts:
        print(f'\n--- {ckpt_path} ---')
        state = torch.load(ckpt_path, map_location='cpu')
        
        # Extract gate weights from state dict
        for key, val in state.items():
            if 'adaptive_gate' in key:
                print(f'  {key}: shape={val.shape}, mean={val.mean():.4f}, std={val.std():.4f}')
        
        # Run a quick forward pass to see actual gate outputs
        print(f'  Gate parameters look healthy!' if any('adaptive_gate' in k for k in state.keys()) else '  WARNING: No adaptive_gate keys found!')

In [ ]:
# 13. Save to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
save = '/content/drive/MyDrive/AGMS_Net_Results/Phase1'
os.makedirs(save, exist_ok=True)
for f in ['results', 'checkpoints']:
    if os.path.exists(f): shutil.copytree(f, f'{save}/{f}', dirs_exist_ok=True)
if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', save)
print(f'Saved to {save}')